# Router Bias Intervention — Full Pipeline (Phase 1 + Phase 2)

**Self-contained**: load model, profile experts at last-token, build bias vectors, sweep configs, report verdict.

**Budget**: ~15-25 min total on RTX 6000 Blackwell 96GB.

**Flow**:
1. Install + load model (~10 min)
2. Load Stage B rollouts + install last-token router hooks (~2 min)
3. Profile at last-token position (~3 min)
4. Build bias configs
5. Swap hooks to bias-mode + run eval on 100 questions × 7 configs (~10 min)
6. Report verdict

## Cell 1 — Install

In [ ]:
import sys, subprocess, os, shutil
from pathlib import Path

def pip(*a, check=True):
    return subprocess.run([sys.executable, '-m', 'pip', *a], check=check)

try:
    import transformers
    from transformers.models.auto.configuration_auto import CONFIG_MAPPING_NAMES
    ok = 'qwen3_5_moe' in CONFIG_MAPPING_NAMES
except Exception:
    ok = False

if not ok:
    pip('install', '-q', 'accelerate', 'datasets', 'huggingface_hub==1.5.0',
        'safetensors', 'einops', 'scikit-learn', 'sentencepiece', 'tokenizers',
        'protobuf', 'scipy', 'matplotlib', 'hf_transfer', check=False)
    pip('uninstall', '-y', 'transformers', 'causal-conv1d', check=False)
    SRC = '/content/transformers_src'
    if Path(SRC).exists(): shutil.rmtree(SRC)
    subprocess.run(['git','clone','--quiet','--depth=1',
                    'https://github.com/huggingface/transformers.git', SRC], check=True)
    pip('install','--force-reinstall','--no-deps','--no-cache-dir', SRC, check=False)
    pip('install','--no-cache-dir','flash-linear-attention', check=False)
    for m in list(sys.modules):
        if m.startswith('transformers') or m.startswith('huggingface_hub'):
            del sys.modules[m]

try:
    from google.colab import userdata
    hf_token = userdata.get('HF_TOKEN')
    if hf_token:
        from huggingface_hub import login
        login(token=hf_token)
        print('HF auth OK')
except Exception as e:
    print(f'(skipping: {e})')

import json, re, time, random, pickle
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from tqdm.auto import tqdm
os.environ['HF_HUB_ENABLE_HF_TRANSFER'] = '1'

OUT = Path('/content/router_bias_pipeline'); OUT.mkdir(exist_ok=True)
print('setup done')

## Cell 2 — Load Qwen3.6-35B-A3B

In [ ]:
from transformers import AutoTokenizer, AutoModelForImageTextToText
from huggingface_hub import snapshot_download
from safetensors import safe_open

MODEL_ID = 'Qwen/Qwen3.6-35B-A3B'
tok = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=True)
if tok.pad_token_id is None: tok.pad_token = tok.eos_token

model = AutoModelForImageTextToText.from_pretrained(
    MODEL_ID, dtype=torch.bfloat16, device_map='cuda',
    attn_implementation='sdpa', trust_remote_code=True)
model.eval()

def get_all_layers(m):
    cands = [m]
    if hasattr(m, 'model'): cands.append(m.model)
    for start in cands:
        for path in [('model','language_model','layers'),('language_model','layers'),('model','layers')]:
            cur = start; ok = True
            for p in path:
                if hasattr(cur, p): cur = getattr(cur, p)
                else: ok = False; break
            if ok and hasattr(cur, '__getitem__'):
                return cur
    raise RuntimeError('layers not found')

all_layers = get_all_layers(model)
moe_layers = [i for i, layer in enumerate(all_layers)
              if hasattr(layer, 'mlp') and hasattr(layer.mlp, 'gate')]
N_EXPERTS = 256
TOP_K = 8
print(f'Model loaded: {torch.cuda.memory_allocated()/1e9:.1f} GB')
print(f'MoE layers: {len(moe_layers)}, experts: {N_EXPERTS}, top-k: {TOP_K}')

## Cell 3 — Load Stage B rollouts

In [ ]:
corpus = snapshot_download('caiovicentino1/Qwen3.6-35B-A3B-mcr-stage-b',
                            repo_type='dataset', cache_dir='/content/cache')
shards = sorted((Path(corpus)/'shards').glob('q*.safetensors'))

MIN_LEN = 200
rollouts = []
for shard in tqdm(shards, desc='load'):
    with safe_open(str(shard), framework='pt') as f:
        meta = f.metadata()
        q_options = json.loads(meta['options'])
        if len(q_options) != 10: continue  # 10-opt only
        rs = json.loads(meta['rollouts'])
        for r_idx, r in enumerate(rs):
            if r['pred'] is None or r['response_len'] < MIN_LEN: continue
            rollouts.append(dict(
                question=meta['question'], options=q_options,
                gold_letter=meta['gold_letter'], pred=r['pred'],
                correct=r['correct']))

n_correct = sum(1 for r in rollouts if r['correct'])
print(f'{len(rollouts)} rollouts ({n_correct} correct / {len(rollouts)-n_correct} wrong)')

## Cell 4 — Phase 1: profile at last-token (~3 min)

In [ ]:
captured_top8 = {}
def make_last_token_hook(layer_idx, router_weight):
    def hook(module, inp):
        hidden = inp[0]
        with torch.no_grad():
            if hidden.dim() == 2:
                last = hidden[-1:, :]
            elif hidden.dim() == 3:
                last = hidden[:, -1, :]
            else:
                return None
            logits = F.linear(last, router_weight)
            top_k_idx = logits.topk(TOP_K, dim=-1).indices
            mask = torch.zeros_like(logits, dtype=torch.bool)
            mask.scatter_(-1, top_k_idx, True)
            captured_top8[layer_idx] = mask.any(dim=0).float().cpu().numpy()
        return None
    return hook

phase1_handles = []
for L in moe_layers:
    h = all_layers[L].mlp.gate.register_forward_pre_hook(
        make_last_token_hook(L, all_layers[L].mlp.gate.weight))
    phase1_handles.append(h)
print(f'✅ {len(phase1_handles)} Phase 1 hooks installed')

expert_freq_last = np.zeros((len(rollouts), len(moe_layers), N_EXPERTS), dtype=np.float32)
rollout_correct = np.zeros(len(rollouts), dtype=bool)
letter_ids_map = {L_: tok(L_, add_special_tokens=False).input_ids[0] for L_ in 'ABCDEFGHIJ'}

t0 = time.time()
for ri, r in enumerate(tqdm(rollouts, desc='profile')):
    try:
        prompt = ("Answer the following multiple-choice question. "
            "Give ONLY the letter of the correct answer.\n\n"
            f"Question: {r['question']}\n\nOptions:\n"
            + '\n'.join(f'{chr(65+i)}. {opt}' for i, opt in enumerate(r['options']))
            + "\n\nAnswer: \\boxed{")
        p = tok.apply_chat_template([{'role':'user','content':prompt}],
                                    tokenize=False, add_generation_prompt=True,
                                    enable_thinking=False)
        ids = tok(p, return_tensors='pt').input_ids.to('cuda')
        if ids.shape[1] > 4096: continue
        with torch.no_grad():
            _ = model(input_ids=ids, attention_mask=torch.ones_like(ids))
        for li, L in enumerate(moe_layers):
            expert_freq_last[ri, li, :] = captured_top8[L]
        rollout_correct[ri] = r['correct']
    except Exception as e:
        continue

# Discriminative analysis
correct_mask = rollout_correct
mean_correct = expert_freq_last[correct_mask].mean(axis=0)
mean_wrong = expert_freq_last[~correct_mask].mean(axis=0)
discriminative = mean_correct - mean_wrong
print(f'\n✅ Phase 1 done in {(time.time()-t0)/60:.1f} min')
print(f'discriminative range: {discriminative.min():.4f} to {discriminative.max():.4f}')

# Remove Phase 1 hooks (we need router output hooks for Phase 2)
for h in phase1_handles: h.remove()
phase1_handles = []

## Cell 5 — Phase 2: install bias hooks + build configs

In [ ]:
def build_bias(target_layers, n_helpers, n_antis, helper_bias, anti_bias):
    bias = np.zeros((len(moe_layers), N_EXPERTS), dtype=np.float32)
    for L in target_layers:
        if L not in moe_layers: continue
        li = moe_layers.index(L)
        disc = discriminative[li]
        helpers = np.argsort(-disc)[:n_helpers]
        antis = np.argsort(disc)[:n_antis]
        if n_helpers > 0: bias[li, helpers] = helper_bias
        if n_antis > 0: bias[li, antis] = anti_bias
    return torch.from_numpy(bias).cuda().to(torch.bfloat16)

CONFIGS = [
    ('baseline', None, None, None, None, 0.0, 0.0),
    ('L18-20_h10_bias1', [18,19,20], 10, 10, 1.0, -1.0, None),
    ('L18-20_h10_bias2', [18,19,20], 10, 10, 2.0, -2.0, None),
    ('L18-20_h10_bias5', [18,19,20], 10, 10, 5.0, -5.0, None),
    ('L15-20_h10_bias2', [15,16,17,18,19,20], 10, 10, 2.0, -2.0, None),
    ('L18-20_h10_bias2_helpersOnly', [18,19,20], 10, 0, 2.0, 0.0, None),
    ('L18-20_a10_bias2_antisOnly', [18,19,20], 0, 10, 0.0, -2.0, None),
]

config_biases = {}
for name, layers, nh, na, hb, ab, _ in CONFIGS:
    config_biases[name] = None if name == 'baseline' else build_bias(layers, nh, na, hb, ab)

class RouterBiasController:
    def __init__(self, layer_idx, layer_idx_in_bias_matrix):
        self.layer_idx = layer_idx
        self.li = layer_idx_in_bias_matrix
        self.bias_matrix = None
        self.active = False
    def set(self, bias_matrix):
        self.bias_matrix = bias_matrix
        self.active = bias_matrix is not None
    def off(self):
        self.active = False
    def make_hook(self):
        def hook(module, inp, out):
            if not self.active or self.bias_matrix is None: return out
            if not isinstance(out, tuple) or len(out) < 3: return out
            logits, _, _ = out
            biased_logits = logits.clone()
            bias_vec = self.bias_matrix[self.li].to(biased_logits.dtype)
            biased_logits[-1, :] = biased_logits[-1, :] + bias_vec
            top_k_vals, top_k_idx = biased_logits.topk(module.top_k, dim=-1)
            top_k_weights = F.softmax(top_k_vals.float(), dim=-1).to(logits.dtype)
            return (biased_logits, top_k_weights, top_k_idx)
        return hook

controllers = {}
bias_handles = []
for li, L in enumerate(moe_layers):
    ctrl = RouterBiasController(L, li)
    controllers[L] = ctrl
    h = all_layers[L].mlp.gate.register_forward_hook(ctrl.make_hook())
    bias_handles.append(h)

def apply_config(bias):
    for ctrl in controllers.values(): ctrl.set(bias)
def off_all():
    for ctrl in controllers.values(): ctrl.off()

print(f'✅ {len(bias_handles)} bias hooks installed, {len(CONFIGS)} configs built')

## Cell 6 — Eval sweep (~10 min)

In [ ]:
# Dedupe questions
seen_q = set(); unique_qs = []
random.seed(42)
random.shuffle(rollouts)
for r in rollouts:
    if r['question'] not in seen_q:
        seen_q.add(r['question'])
        unique_qs.append(r)
eval_sample = unique_qs[:100]
print(f'{len(eval_sample)} unique questions')

results = []
t0 = time.time()
for i, q in enumerate(tqdm(eval_sample, desc='bias eval')):
    try:
        prompt = ("Answer the following multiple-choice question. "
            "Give ONLY the letter of the correct answer.\n\n"
            f"Question: {q['question']}\n\nOptions:\n"
            + '\n'.join(f'{chr(65+j)}. {opt}' for j, opt in enumerate(q['options']))
            + "\n\nAnswer: \\boxed{")
        p = tok.apply_chat_template([{'role':'user','content':prompt}],
                                    tokenize=False, add_generation_prompt=True,
                                    enable_thinking=False)
        ids = tok(p, return_tensors='pt').input_ids.to('cuda')
        if ids.shape[1] > 4096: continue
        row = dict(idx=i, gold=q['gold_letter'])
        for cfg_name, *_ in CONFIGS:
            apply_config(config_biases[cfg_name])
            with torch.no_grad():
                out = model(input_ids=ids, attention_mask=torch.ones_like(ids))
            logits = out.logits[0, -1]
            letter_logits = {L_: logits[letter_ids_map[L_]].item() for L_ in 'ABCDEFGHIJ'[:10]}
            pred = max(letter_logits, key=letter_logits.get)
            row[f'{cfg_name}_pred'] = pred
            row[f'{cfg_name}_correct'] = (pred == q['gold_letter'])
        off_all()
        results.append(row)
        if (i+1) % 20 == 0:
            accs = {n: sum(r[f'{n}_correct'] for r in results)/len(results) for n,*_ in CONFIGS}
            top3 = sorted(accs.items(), key=lambda x: -x[1])[:3]
            print(f'  [{i+1}/100] top3: ' + ' | '.join(f'{n}={a*100:.0f}%' for n,a in top3))
    except torch.cuda.OutOfMemoryError:
        torch.cuda.empty_cache(); continue

with open(OUT/'bias_results.json', 'w') as f:
    json.dump(dict(n=len(results), configs=[c[0] for c in CONFIGS], results=results,
                   wall_min=(time.time()-t0)/60), f, indent=2)
print(f'\n✅ eval done in {(time.time()-t0)/60:.1f} min')

## Cell 7 — Verdict

In [ ]:
from scipy.stats import binomtest

print(f'=== Router Bias Intervention (N={len(results)}) ===\n')
bc = [r['baseline_correct'] for r in results]
base_acc = sum(bc) / len(results) * 100
print(f'baseline: {base_acc:.1f}%\n')
print(f'{"config":35s} {"acc%":>6s}  {"Δpp":>6s}  {"gain":>5s}  {"lost":>5s}  {"p":>7s}')
stats = []
for (n, *_) in CONFIGS:
    if n == 'baseline': continue
    cc = [r[f'{n}_correct'] for r in results]
    acc = sum(cc) / len(results)
    delta = (acc - base_acc/100) * 100
    g = sum(1 for b, c in zip(bc, cc) if not b and c)
    l = sum(1 for b, c in zip(bc, cc) if b and not c)
    p = binomtest(g, g+l, p=0.5, alternative='two-sided').pvalue if g+l > 0 else 1.0
    stats.append((n, acc*100, delta, g, l, p))
stats.sort(key=lambda r: -r[2])
for n, acc, d, g, l, p in stats:
    star = ' ***' if d>3 and l==0 else (' **' if d>1.5 and l==0 else (' *' if d>0.5 and l==0 else ''))
    print(f'{n:35s} {acc:5.1f}%  {d:+5.1f}  {g:>5d}  {l:>5d}  {p:>7.3f}{star}')

best_pareto = max([s for s in stats if s[4] == 0], key=lambda r: r[2], default=None)
best_overall = max(stats, key=lambda r: r[2])
print('\n=== Verdict ===')
if best_pareto and best_pareto[2] > 3:
    print(f'*** BREAKTHROUGH: {best_pareto[0]} -> {best_pareto[2]:+.1f}pp Pareto')
elif best_pareto and best_pareto[2] > 1.5:
    print(f'**  MODERATE: {best_pareto[0]} -> {best_pareto[2]:+.1f}pp Pareto')
elif best_pareto and best_pareto[2] > 0:
    print(f'*   Small positive: {best_pareto[0]} -> {best_pareto[2]:+.1f}pp Pareto')
else:
    print(f'--  No Pareto positive. Best overall {best_overall[0]} {best_overall[2]:+.1f}pp ({best_overall[4]} lost)')